# <font color = 'red'> DEPENDENCIAS

In [1]:
import pandas as pd
import numpy as np
import plotly.express as px
import statsmodels.api as sm
import statsmodels.formula.api as smf
from statsmodels.miscmodels.ordinal_model import OrderedModel
from sklearn.preprocessing import MinMaxScaler

import sys
import os

# Agregar la carpeta calibration_code al path
sys.path.append(os.path.abspath("../../calibration_code"))

# Ahora puedes importar los módulos personalizados
from modelling_tools import (plot_histogram, plot_univariate_freq, assign_deciles, count_categories_by_decile, 
                             calculate_category_proportions, summarize_decile_analysis, summarize_grouped_deciles, group_deciles,
                             compute_odds_ratio)
from visualization_tools import plot_interactive_chart
from utils import g
from config import get_data_path, get_code_path
from data_cleaning import check_dataframe_quality

# <font color = 'red'> CARGA DE DATOS

In [2]:
df = pd.read_csv(get_data_path("bivariate_preprocessed_data.csv"))

In [3]:
res = check_dataframe_quality(df)

No missing values found.
No infinite values found.
No duplicate rows found.


# <font color = 'red'> ANÁLISIS

In [6]:
col = "Num_Credit_Card"

## <font color = 'skyblue'> ANÁLISIS GENERAL

Las medianas tienen el orden esperado: Median Bad > Mediana Standard > Mediana Good

In [7]:
fig_box = px.box(df, x="Credit_Mix", y=col, title=f"Distribution of {col} by Credit Score Category")
fig_box.show()

## <font color = 'skyblue'> ANÁLISIS POR DECILES

In [8]:
continuous_variable= col
decile_col_name = continuous_variable + '_Decile'
target_col_string = "Credit_Mix" # variable dependiente con nombres string
target_col = 'Credit_Score' # variable dependiente int (para modelos)

In [9]:
analysis_summary = summarize_decile_analysis(df, continuous_variable, decile_col_name, target_col_string)

# Obtener los resultados
df_deciles = analysis_summary["df_deciles"]  # DataFrame con los deciles asignados
deciles_summary = analysis_summary["decile_summary"]  # Resumen de deciles con conteos y proporciones
display(deciles_summary)
res = check_dataframe_quality(df_deciles)

,Decile_Min,Decile_Max,Decile_Count,Decile_Proportion,count_Bad,count_Good,count_Standard,prop_Bad,prop_Good,prop_Standard
Num_Credit_Card_Decile,,,,,,,,,,
0,0,3,17957,0.17957,0,10018,7939,0.000000,0.557888,0.442112
1,4,4,14358,0.14358,29,6071,8258,0.002020,0.422830,0.575150
2,5,5,18904,0.18904,3997,6144,8763,0.211437,0.325011,0.463553
3,6,6,16929,0.16929,3985,3962,8982,0.235395,0.234036,0.530569
4,7,7,17029,0.17029,4057,4146,8826,0.238241,0.243467,0.518292
5,8,8,5072,0.05072,4017,43,1012,0.791995,0.008478,0.199527
6,9,11,9751,0.09751,7683,0,2068,0.787919,0.000000,0.212081


No missing values found.
No infinite values found.
No duplicate rows found.


In [10]:
df[(df[continuous_variable] >= 6) & (df[continuous_variable] <= 6) & (df['Credit_Score'] == 2)].shape

(3962, 85)

Porporción de Buenos: aunque se observa una relación positiva entre los ingresos netos y la proporción de buenos, se observa un
cambio abrupto entre los deciles 6 y 7, al pasar de una proporción de buenos de 41% a 15%,
lo cual no es razonable.

Proporción de standard: aunque sí existe una tendencia negativa entre la proporción de Standard y los ingresos
hay cambios irregulares que son poco razonables.

Proporción de malos: de manera similar, la proporción de malos y los ingresos tienen una relación inversa, sin embargo, 
hay cambios irregulares abruptos.

In [11]:
chart_types = {
    "prop_Good":"line",
    "prop_Standard":"line",
    "prop_Bad": "line",  
    "Decile_Count": "bar"    
}

fig = plot_interactive_chart(
    df=deciles_summary,  
    y_columns=["prop_Bad", "prop_Good", "prop_Standard", "Decile_Count"],  
    x_column="Decile_Max",  
    chart_types=chart_types, 
    title=f"Proportion of Credit Score Categories by {continuous_variable}",
    x_title="Decile",
    y_title="Decile Count",  
    y2_title="Proportion",   
    secondary_y=["prop_Bad", "prop_Good", "prop_Standard"],  
    width=900,
    height=500,
    custom_colors={"prop_Bad": "red", "prop_Good":"lightgreen",
    "prop_Standard":"brown", "Decile_Count": "gray"}  
)

fig.show()

<font color = 'brown'> Agrupación de deciles

Se agrupan deciles buscando una relación monótona entre las proporciones y los ingresos:

In [12]:
group_map = {0: "Group_1", 
             1: "Group_1", 
             2: "Group_2", 
             3: "Group_2", 
             4: "Group_2",
             5: "Group_3", 
             6: "Group_3"}

# Agrupar los deciles
grouped_col_name = "Grouped_" + continuous_variable

df_deciles_grouped = group_deciles(df_deciles, decile_col_name, grouped_col_name, group_map)

summary_results = summarize_grouped_deciles(df_deciles_grouped, grouped_col_name, continuous_variable, target_col_string, prefix="Decile_")
grouped_deciles_summary = summary_results['df']


# Definir el mapeo manual de los grupos a enteros
group_mapping = {
    'Group_1': 1,
    'Group_2': 2,
    'Group_3': 3
}

if not set(group_map.values()) == set(group_mapping.keys()):
    print("Problemas en el mapeo de grupos a enteros!")

# Usar `.map()` en lugar de `.replace()` para evitar el warning
df_deciles_grouped[grouped_col_name] = (
    df_deciles_grouped[grouped_col_name]
    .map(group_mapping)  # Mapear los valores
    .astype("Int64")      # Convertir a entero manejando NaN si existen
)

display(grouped_deciles_summary)

res = check_dataframe_quality(df_deciles_grouped)

,Decile_Min,Decile_Max,Decile_Count,Decile_Proportion,count_Bad,count_Good,count_Standard,prop_Bad,prop_Good,prop_Standard
Grouped_Num_Credit_Card,,,,,,,,,,
Group_1,0,4,32315,0.32315,29,16089,16197,0.000897,0.497880,0.501222
Group_2,5,7,52862,0.52862,12039,14252,26571,0.227744,0.269608,0.502648
Group_3,8,11,14823,0.14823,11700,43,3080,0.789314,0.002901,0.207785


No missing values found.
No infinite values found.
No duplicate rows found.


In [13]:
chart_types = {
    "prop_Good":"line",
    "prop_Standard":"line",
    "prop_Bad": "line",  
    "Decile_Count": "bar"    
}

fig = plot_interactive_chart(
    df=grouped_deciles_summary,  
    y_columns=["prop_Bad", "prop_Good", "prop_Standard", "Decile_Count"],  
    x_column="Decile_Max",  
    chart_types=chart_types, 
    title=f"Proportion of Credit Score Categories by {continuous_variable}",
    x_title="Decile",
    y_title="Decile Count",  
    y2_title="Proportion",   
    secondary_y=["prop_Bad", "prop_Good", "prop_Standard"],  
    width=900,
    height=500,
    custom_colors={"prop_Bad": "red", "prop_Good":"lightgreen",
    "prop_Standard":"brown", "Decile_Count": "gray"}  
)

fig.show()

## <font color = 'skyblue'> REGRESIONES BIVARIADAS

Como Credit_Score tiene tres categorías (Bad, Standard, Good) se pueden usar dos enfoques 
de regresión categórica: 

- Regresión Logística Multinomial → No asume orden en las categorías (como si fueran colores: rojo, azul, verde).
- Regresión Logística Ordinal → Asume que hay un orden en las categorías (Bad < Standard < Good).

Dado que hay un orden entre las categorías se utiliza Regresión Logística Ordinal:

<font color = 'gold'> Sin Agrupaciones

In [14]:
df_ = df_deciles_grouped.copy()
x_variable = continuous_variable
res = check_dataframe_quality(df_)

No missing values found.
No infinite values found.
No duplicate rows found.


In [15]:
# para no generar problemas numéricos debe escalarse esta variable.
# se opta por normalizar la variable:

g(df_[[x_variable]].describe()).transpose()

,count,mean,std,min,25%,50%,75%,max
Num_Credit_Card,"100,000.00",5.53,2.07,0.00,4.00,5.00,7.00,11.00


Todos los coeficientes son significativos.

Num_Credit_Card_Scaled -6.6449: según lo esperado, el coeficiente es negativo: por cada número de crédito adicional, la probabilidad de estar en una categoría superior de Credit_Score disminuye.

Threshold 0/1 -4.8075: Umbral que separa las categorías Bad y Standard. Si la puntuación supera este umbral, es más probable que 
sea standard en lugar de bad.

Threshold 1/2 0.9557: Umbral que separa las categorías Standard y Good. Si la puntuación supera este umbral, es más probable que 
sea Good en lugar de Standard.

In [17]:
df_ = df_deciles_grouped.copy()
x_variable = continuous_variable

scaler = MinMaxScaler()
df_[continuous_variable + "_Scaled"] = scaler.fit_transform(df_[[x_variable]])

res = check_dataframe_quality(df_)

# Ajustar el modelo de regresión logística ordinal con la variable escalada
model_income = OrderedModel(df_[target_col], df_[continuous_variable + "_Scaled"], distr="logit")
result_income = model_income.fit(method='bfgs')

# Mostrar resumen del modelo
print(result_income.summary())

# Calcular e interpretar el Odds Ratio
res_odds = compute_odds_ratio(result_income, variable_name=continuous_variable + "_Scaled", description="Ingreso Anual Escalado")
print(res_odds["interpretation"])


No missing values found.
No infinite values found.
No duplicate rows found.
Optimization terminated successfully.
         Current function value: 0.890005
         Iterations: 12
         Function evaluations: 14
         Gradient evaluations: 14
                             OrderedModel Results                             
Dep. Variable:           Credit_Score   Log-Likelihood:                -89000.
Model:                   OrderedModel   AIC:                         1.780e+05
Method:            Maximum Likelihood   BIC:                         1.780e+05
Date:                Sat, 29 Mar 2025                                         
Time:                        18:28:01                                         
No. Observations:              100000                                         
Df Residuals:                   99997                                         
Df Model:                           1                                         
                             coef    std 

<font color = 'gold'> Por Deciles

Todos los coeficientes son significativos.

Num_Credit_Card_Decile -0.6341: Por cada decil adicional, la probabilidad de estar en una categoría superior de Credit_Score disminuye.

Threshold 0/1 -3.0622: Umbral que separa las categorías Bad y Standard. Si la puntuación supera este umbral, es más probable que 
sea standard en lugar de bad.

Threshold 1/2 0.9456: Umbral que separa las categorías Standard y Good. Si la puntuación supera este umbral, es más probable que 
sea Good en lugar de Standard.

In [18]:
df_ = df_deciles_grouped.copy()
x_variable = decile_col_name

res = check_dataframe_quality(df_)

model_age_decile = OrderedModel(df_[target_col], df_[x_variable], distr="logit")

result_age_decile = model_age_decile.fit(method='bfgs')

print(result_age_decile.summary())

res_odds = compute_odds_ratio(result_age_decile, variable_name = x_variable, description = 'Decil de Ingreso Neto')
print(res_odds['interpretation'])

No missing values found.
No infinite values found.
No duplicate rows found.
Optimization terminated successfully.
         Current function value: 0.902806
         Iterations: 10
         Function evaluations: 12
         Gradient evaluations: 12
                             OrderedModel Results                             
Dep. Variable:           Credit_Score   Log-Likelihood:                -90281.
Model:                   OrderedModel   AIC:                         1.806e+05
Method:            Maximum Likelihood   BIC:                         1.806e+05
Date:                Sat, 29 Mar 2025                                         
Time:                        18:32:45                                         
No. Observations:              100000                                         
Df Residuals:                   99997                                         
Df Model:                           1                                         
                             coef    std 

<font color = 'gold'> Por Agrupamientos de Deciles

Todos los coeficientes son significativos.

Grouped_Num_Credit_Card -1.8020: Por cada grupo adcional, la probabilidad de estar en una categoría superior de Credit_Score disminuye.

Threshold 0/1 -4.7324: Umbral que separa las categorías Bad y Standard.

Threshold 1/2 0.9516: Umbral que separa las categorías Standard y Good. Si la puntuación supera 0.7604, es más probable que 
sea Good en lugar de Standard.

In [19]:
df_ = df_deciles_grouped.copy()
x_variable = grouped_col_name

df_[target_col] = df_[target_col].astype(int)
df_[x_variable] = df_[x_variable].astype(int)

res = check_dataframe_quality(df_)

model_age_decile = OrderedModel(df_[target_col], df_[x_variable], distr="logit")

result_age_decile = model_age_decile.fit(method='bfgs')

print(result_age_decile.summary())

res_odds = compute_odds_ratio(result_age_decile, variable_name = x_variable, description = 'Decil de Edad')
print(res_odds['interpretation'])

No missing values found.
No infinite values found.
No duplicate rows found.
Optimization terminated successfully.
         Current function value: 0.895402
         Iterations: 11
         Function evaluations: 12
         Gradient evaluations: 12
                             OrderedModel Results                             
Dep. Variable:           Credit_Score   Log-Likelihood:                -89540.
Model:                   OrderedModel   AIC:                         1.791e+05
Method:            Maximum Likelihood   BIC:                         1.791e+05
Date:                Sat, 29 Mar 2025                                         
Time:                        18:34:11                                         
No. Observations:              100000                                         
Df Residuals:                   99997                                         
Df Model:                           1                                         
                              coef    std

## <font color = 'skyblue'> CONCLUSIONES

%%markdown

### 📊 Comparación de Representaciones de `Num_Credit_Card` usando Regresión Ordinal

| Representación                    | Coeficiente principal | Indicadores de ajuste                                                                 | Interpretación                                                                 |
|----------------------------------|------------------------|----------------------------------------------------------------------------------------|--------------------------------------------------------------------------------|
| `Num_Credit_Card_Scaled`         | -6.6449                | **Log-Likelihood**: -89,000<br>**AIC**: 178,000<br>**BIC**: 178,041                    | 🔹 Mejor ajuste.<br>🔹 El modelo conserva mayor detalle al usar la variable continua escalada. |
| `Num_Credit_Card_Decile`         | -0.6341                | **Log-Likelihood**: -90,281<br>**AIC**: 180,562<br>**BIC**: 180,603                    | 🔹 Peor ajuste.<br>🔹 La discretización reduce capacidad explicativa. |
| `Grouped_Num_Credit_Card`        | -1.8020                | **Log-Likelihood**: -89,540<br>**AIC**: 179,081<br>**BIC**: 179,122                    | 🔹 Ajuste intermedio.<br>🔹 Agrupar deciles mejora la interpretación sin sacrificar tanto el ajuste. |


## <font color = 'skyblue'> EXPORTACIÓN DE DATOS CON VARIABLES ADICIONALES

In [42]:
# df_deciles_grouped.to_csv("../../calibration_data/preprocessed_data.csv", index=False)
